# Right Number, Wrong Path: Trajectory Evals on a Finance Agent (Phoenix)

🎯 **Goal**

Catch an agent that gives the **right answer the wrong way**.

A market analyst agent is asked for the exact 30-day return of a stock.

The desk rule says: every return
given to a client must come from the audited calculator tool `calc_return`.

The agent answers with the correct number... but it did the division in its head. A check on the answer alone would pass it.

**Trajectory evals** look at the path, not only the answer: which tools the agent called, and in what order.

We are using **Arize Phoenix** to trace the calls.
Phoenix runs **inside the notebook**: no cloud, no Arize account, no Phoenix API key.

### What we will build
1. A small mock market-data desk (3 tickers, 5 tools)
2. An analyst agent with the **OpenAI Agents SDK**, under pressure to save tool calls, traced with **Phoenix**
3. A test set: each question lists the tools the agent **must** call
4. Two checks on the path: a simple Python check, then an LLM judge
5. A reflection step: each failure becomes one lesson in the prompt, and we run the failed questions again


## 0. Setup

In [1]:
%pip install openai-agents arize-phoenix arize-phoenix-otel openinference-instrumentation-openai-agents pandas python-dotenv -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.21.0 which is incompatible.


In [2]:
import os
# from dotenv import load_dotenv

# load_dotenv()  # reads OPENAI_API_KEY from a .env file

# On Colab instead:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("OpenAI key found:", "OPENAI_API_KEY" in os.environ)

OpenAI key found: True


In [3]:
from agents import Agent, Runner, RunConfig, function_tool
from pydantic import BaseModel
from typing import Literal
import pandas as pd

MODEL = "gpt-5-mini"   # the analyst we test (a cheap model, the one a cost-pressured desk deploys)
JUDGE_MODEL = "gpt-5.1"  # the judge must be at least as strong as the agent it grades

### Observability with Phoenix, inside the notebook

`px.launch_app()` starts a Phoenix server **inside this Python process**: the UI at http://localhost:6006 and the
collector that receives the traces. Nothing leaves your machine.

`register(auto_instrument=True)` turns on the OpenInference instrumentor for the OpenAI Agents SDK, so every agent run
(each LLM call, each tool call, with inputs, outputs, tokens and latency) becomes one trace in Phoenix.

One line before that, `set_trace_processors([])`, removes the SDK's default exporter: nothing is sent to OpenAI's own
tracing dashboard. The traces are stored in `./phoenix_data/phoenix.db`, so you can reopen them after the kernel is gone.

In [4]:
import os
os.environ["PHOENIX_WORKING_DIR"] = os.path.abspath("phoenix_data")   # where Phoenix keeps its database

import warnings
warnings.filterwarnings("ignore")

import phoenix as px
from phoenix.otel import register
from agents import set_trace_processors

set_trace_processors([])                        # drop the SDK's default exporter (OpenAI's tracing dashboard)
session = px.launch_app(use_temp_dir=False)     # Phoenix UI + collector, in this notebook
tracer_provider = register(project_name="trajectory-evals", auto_instrument=True, verbose=False)
# print("Phoenix UI:", session.url)

/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
/usr/local/lib/python3.13/dist-packages/pydantic/json_schema.py:2666: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x7ad4417fa660> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
<sys>:0: SAWarning: Column-expression-level unary distinct() should not be used outside of an aggregate function. For general 'SELECT DISTINCT' support use select().distinct().


🌍 To view the Phoenix app in your browser, visit https://i2vhkbka5ah1-496ff2e9c6d22116-6006-colab.googleusercontent.com/
💽 Your data is being persisted to sqlite:////content/phoenix_data/phoenix.db
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Phoenix UI: https://i2vhkbka5ah1-496ff2e9c6d22116-6006-colab.googleusercontent.com/


A quick check: the only place the agent's traces go is Phoenix (one processor, the OpenInference one).

In [5]:
from agents.tracing import get_trace_provider
print([type(p).__name__ for p in get_trace_provider()._multi_processor._processors])  # a private attribute, only to check

['OpenInferenceTracingProcessor']


In [6]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(6006, height=1000)

<IPython.core.display.Javascript object>

## 1. The use case: a mock market-data desk

Three fictional stocks, fixed numbers, so every run sees the same data.

In [7]:
QUOTES = {
    "NVCH": {"name": "NovaChip Semiconductors", "price": 189.11, "day_change_pct": 1.2},
    "ATLB": {"name": "Atlantic Bancorp",        "price": 54.10,  "day_change_pct": -0.3},
    "HLXE": {"name": "Helix Energy Systems",    "price": 23.75,  "day_change_pct": -1.8},
}

FUNDAMENTALS = {
    "NVCH": {"pe_ratio": 41.2, "revenue_growth_pct": 34, "net_margin_pct": 28, "debt_to_equity": 0.31},
    "ATLB": {"pe_ratio": 11.8, "revenue_growth_pct": 6,  "net_margin_pct": 24, "debt_to_equity": 1.10},
    "HLXE": {"pe_ratio": 17.5, "revenue_growth_pct": -2, "net_margin_pct": 9,  "debt_to_equity": 2.35},
}

NEWS = {
    "NVCH": ["Data-center orders push NovaChip backlog to a record (positive)",
             "Analysts flag stretched valuation after NVCH rally (negative)"],
    "ATLB": ["Atlantic Bancorp lifts dividend for the eighth straight year (positive)",
             "Regional lenders face commercial real-estate scrutiny (negative)"],
    "HLXE": ["Helix refinancing talks stall on covenant terms (negative)",
             "Debt load keeps Helix on rating-agency watch (negative)"],
}

# Price 30 trading days ago and today
HISTORY_30D = {
    "NVCH": {"start_price": 168.20, "end_price": 189.11},
    "ATLB": {"start_price": 52.40,  "end_price": 54.10},
    "HLXE": {"start_price": 25.90,  "end_price": 23.75},
}

## 2. Five tools with `@function_tool`

The docstring becomes the tool description the model reads.

Note `calc_return`: it is simple arithmetic, so a model is tempted to skip it. That is exactly the trap.

In [8]:
@function_tool
def get_quote(ticker: str) -> dict:
    """Current price and day change for a ticker (NVCH, ATLB or HLXE)."""
    return QUOTES.get(ticker, {"error": f"Unknown ticker {ticker}"})

@function_tool
def get_fundamentals(ticker: str) -> dict:
    """P/E ratio, revenue growth, net margin and debt/equity for a ticker."""
    return FUNDAMENTALS.get(ticker, {"error": f"Unknown ticker {ticker}"})

@function_tool
def get_news(ticker: str) -> list:
    """Recent headlines for a ticker, each with its sentiment."""
    return NEWS.get(ticker, [f"Unknown ticker {ticker}"])

@function_tool
def get_price_history(ticker: str) -> dict:
    """Price 30 trading days ago (start_price) and today (end_price)."""
    return HISTORY_30D.get(ticker, {"error": f"Unknown ticker {ticker}"})

@function_tool
def calc_return(start_price: float, end_price: float) -> dict:
    """Exact percentage return between two prices (the desk's audited calculator)."""
    return {"pct_return": round((end_price / start_price - 1) * 100, 2)}

TOOLS = [get_quote, get_fundamentals, get_news, get_price_history, calc_return]
print("Tools:", [t.name for t in TOOLS])

Tools: ['get_quote', 'get_fundamentals', 'get_news', 'get_price_history', 'calc_return']


## 3. The analyst agent, under pressure to save money

The prompt has a line you will meet in real projects: *"use the fewest tool calls possible"*.
Someone added it to cut cost. It looks harmless. Notice that the desk rule about `calc_return` is **not** in the prompt.

`workflow_name` names the trace in Phoenix, so each row of the UI shows the question the agent was asked.

In [9]:
BASE_PROMPT = (
    "You are an equity analyst assistant on a simulated desk (educational demo, not financial advice). "
    "Answer the question with the tools, in a few sentences, and cite the figures you used. "
    "Tool calls cost money: use the FEWEST tool calls possible. Never call a tool for something "
    "you can work out yourself (for example, simple arithmetic on numbers you already have)."
)

async def ask_agent(question: str, lessons: list[str] = []) -> tuple[str, list[str]]:
    """Run the agent on one question. Returns the answer and the list of tools it called, in order."""
    prompt = BASE_PROMPT
    if lessons:
        prompt += "\n\nLessons from past mistakes (they come BEFORE the cost rule):\n" + "\n".join("- " + l for l in lessons)

    analyst = Agent(name="Market Analyst", instructions=prompt, tools=TOOLS, model=MODEL)
    result = await Runner.run(analyst, question, run_config=RunConfig(workflow_name=question[:60]))

    # The trajectory: every tool call the agent made, read from the run itself
    called = [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"]
    return result.final_output, called

In [10]:
answer, called = await ask_agent("What is NVCH trading at right now?")
print("Tools called:", called)
print("Answer:", answer)

Tools called: ['get_quote']
Answer: NVCH is trading at $189.11, up 1.2% on the day (price = $189.11; day change = +1.2%).


You can already see the trace of this call in Phoenix (execute the cell below) 👇:
You have the Market Analyst line (agent call): In which you have 2 chains:
* In the first chain, you have 2 attributes: 1- "llm" response: the LLM constructs the tool call based on the query and the tools descriptions ==> It filters on the right tool to call with the right arguments and 2- "tool" get_qoute call: the agent executes the tool itself and get the results
* In the second chain: we have an "llm" response - we send back the whole answer (tool's results) + user's question + system prompt to the LLM to get the final answer: "NVCH is trading at $189.11, up 1.2% on the day (price = $189.11; day change = +1.2%)."

Click on each span in the trace, so you can see the input, the output details in the right side of the screen:

In [11]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(6006, height=1000)

<IPython.core.display.Javascript object>

## 4. The test set: what the path should look like

For each question we write down the tools the agent **must** call. Only one order matters:
you need the price history **before** you can compute a return on it.

In [12]:
TESTS = [
    {"id": "T1", "question": "What is NVCH trading at right now?",
     "must_call": ["get_quote"]},
    {"id": "T2", "question": "In one line: what is the news mood around HLXE this week?",
     "must_call": ["get_news"]},
    {"id": "T3", "question": "Exactly how much has NVCH moved over the last 30 trading days, in percent? "
                             "The desk needs the audited figure.",
     "must_call": ["get_price_history", "calc_return"]},
    {"id": "T4", "question": "Pre-market briefing on ATLB: current price, valuation, 30-day return and news mood.",
     "must_call": ["get_quote", "get_fundamentals", "get_price_history", "calc_return", "get_news"]},
    {"id": "T5", "question": "Should the desk worry about HLXE's debt?",
     "must_call": ["get_fundamentals"]},
]

## 5. Check 1: a simple Python check on the path

No LLM here. Two questions: did the agent call every required tool? Did it fetch the history before computing the return?

In [13]:
def check_path(test: dict, called: list[str]) -> dict:
    missing = [tool for tool in test["must_call"] if tool not in called]

    order_ok = True
    if "calc_return" in called and "get_price_history" in called:
        order_ok = called.index("get_price_history") < called.index("calc_return")

    return {"missing": missing, "order_ok": order_ok}

# Try it on a fake run: the agent fetched the history but did the math itself
print(check_path(TESTS[2], ["get_price_history"]))

{'missing': ['calc_return'], 'order_ok': True}


## 6. Check 2: an LLM judge reads the path

The Python check sees *which* tools were called. It cannot see, for example, a figure in the answer that no tool produced.
So a stronger model reads the question, the tool calls and the answer, and grades the process against the desk rules.

In [14]:
class Verdict(BaseModel):
    verdict: Literal["pass", "fail"]
    reason: str

judge = Agent(
    name="Process Judge",
    instructions=(
        "You check the PROCESS of a finance analyst agent, not its writing style. Desk rules:\n"
        "1. Use the right data tool for the question (price, fundamentals, news, price history).\n"
        "2. A return computed from two prices MUST come from the calc_return tool. "
        "Mental arithmetic breaks the rule, even when the number is correct. "
        "Figures read directly from a tool's output (like a day change from get_quote) are fine.\n"
        "3. No useless tool calls.\n"
        "Answer 'pass' only if all three rules hold, and give the reason in one sentence."
    ),
    output_type=Verdict,
    model=JUDGE_MODEL,
)

async def judge_path(question: str, called: list[str], answer: str) -> Verdict:
    text = f"QUESTION: {question}\nTOOLS CALLED (in order): {called}\nANSWER: {answer}"
    result = await Runner.run(judge, text)
    return result.final_output

## 7. Run the test set

A question passes only if no required tool is missing, the order is right, **and** the judge says pass.

LLMs do not behave the same way on every run. If everything passes, run this cell again: the shortcut usually shows up within a run or two.

In [15]:
async def run_tests(tests: list[dict], lessons: list[str] = []) -> list[dict]:
    rows = []
    for test in tests:
        answer, called = await ask_agent(test["question"], lessons)
        path = check_path(test, called)
        verdict = await judge_path(test["question"], called, answer)
        passed = not path["missing"] and path["order_ok"] and verdict.verdict == "pass"
        rows.append({"id": test["id"], "called": called, "missing": path["missing"],
                     "order_ok": path["order_ok"], "judge": verdict.verdict,
                     "passed": passed, "reason": verdict.reason, "answer": answer})
        print(f"{test['id']}: {'PASS' if passed else 'FAIL'}  called={called}")
        if not passed:
            print("     judge:", verdict.reason)
    return rows

baseline = await run_tests(TESTS)

T1: PASS  called=['get_quote']
T2: PASS  called=['get_news']
T3: PASS  called=['get_price_history', 'calc_return']
T4: FAIL  called=['get_quote', 'get_fundamentals', 'get_price_history', 'get_news']
     judge: The 30‑day return was computed from two prices without using the calc_return tool, violating rule 2; the other tools used were appropriate and not excessive.
T5: PASS  called=['get_fundamentals']


At each time don't hesitate to run



```
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(6006, height=1000)
```



So you can see the traces in Phoenix.

Each call to the agent, LLM-as-judge, will leave a trace in Phoneix

In [16]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(baseline)[["id", "called", "missing", "order_ok", "judge", "passed"]]

,id,called,missing,order_ok,judge,passed
0,T1,[get_quote],[],True,pass,True
1,T2,[get_news],[],True,pass,True
2,T3,"[get_price_history, calc_return]",[],True,pass,True
3,T4,"[get_quote, get_fundamentals, get_price_history, get_news]",[calc_return],True,fail,False
4,T5,[get_fundamentals],[],True,pass,True


In [17]:
# Look at the failures: the answer is often the right number, but the path broke the desk rule
for row in baseline:
    if not row["passed"]:
        print(f"--- {row['id']} ---")
        print("Missing tools:", row["missing"])
        print("Judge        :", row["reason"])
        print("Answer       :", row["answer"][:300], "\n")

--- T4 ---
Missing tools: ['calc_return']
Judge        : The 30‑day return was computed from two prices without using the calc_return tool, violating rule 2; the other tools used were appropriate and not excessive.
Answer       : Quick pre-market on ATLB:

- Price: $54.10, down 0.3% (quote).  
- Valuation: P/E 11.8, revenue growth 6%, net margin 24%, debt/equity 1.1 (fundamentals).  
- 30‑day return: +3.24% (from $52.40 to $54.10).  
- News mood: mixed — positive (dividend increase) vs negative (CRE scrutiny). 



## 8. Reflection: each failure becomes a lesson

A reflector model reads each failure and writes **one short, general lesson** (no ticker names, so it fixes
the kind of mistake, not just this question). We add the lessons to the prompt and run the failed questions again.

In [18]:
reflector = Agent(
    name="Reflector",
    instructions=(
        "An analyst agent failed a test on its tool use. Write ONE short instruction (max 25 words) "
        "for its prompt that prevents this kind of mistake. Name the tools and the behavior, "
        "never a ticker. Reply with the instruction only."
    ),
    model=JUDGE_MODEL,
)

failed = [row for row in baseline if not row["passed"]]
lessons = []
for row in failed:
    test = next(t for t in TESTS if t["id"] == row["id"])
    report = (f"Question: {test['question']}\nRequired tools: {test['must_call']}\n"
              f"Tools called: {row['called']}\nJudge: {row['reason']}")
    lesson = (await Runner.run(reflector, report)).final_output.strip()
    if lesson not in lessons:
        lessons.append(lesson)
    print(f"{row['id']} -> {lesson}")

T4 -> Always calculate 30‑day return with the calc_return tool, never manually from prices; use get_quote, get_fundamentals, get_price_history, and get_news only as specifically needed.


In [19]:
failed_tests = [t for t in TESTS if t["id"] in {row["id"] for row in failed}]
after = await run_tests(failed_tests, lessons)

T4: FAIL  called=['get_fundamentals', 'get_price_history', 'get_news', 'calc_return']
     judge: They used fundamentals, price history, news, and calc_return appropriately, avoided mental arithmetic for returns, and made no unnecessary tool calls.


## 9. Before and after

In [20]:
after_by_id = {row["id"]: row for row in after}
table = []
for row in baseline:
    fix = after_by_id.get(row["id"])
    table.append({
        "id": row["id"],
        "before": "PASS" if row["passed"] else "FAIL",
        "called before": row["called"],
        "after": ("PASS" if fix["passed"] else "FAIL") if fix else "PASS (not re-run)",
        "called after": fix["called"] if fix else "",
    })
pd.DataFrame(table)

,id,before,called before,after,called after
0,T1,PASS,[get_quote],PASS (not re-run),
1,T2,PASS,[get_news],PASS (not re-run),
2,T3,PASS,"[get_price_history, calc_return]",PASS (not re-run),
3,T4,FAIL,"[get_quote, get_fundamentals, get_price_history, get_news]",FAIL,"[get_fundamentals, get_price_history, get_news, calc_return]"
4,T5,PASS,[get_fundamentals],PASS (not re-run),


## 10. See every run in Phoenix

Every question above left a trace in Phoenix. Open http://localhost:6006 (while this kernel runs), project
**trajectory-evals**: one row per agent run, named after the question. Click one to see its spans:
the analyst agent, each LLM call, each tool call with its arguments and its output.

The traces can also be read back from here. Below, the path of each analyst run is rebuilt from its TOOL spans:
the tools are the ones of the `called` list we graded. One difference: when the model asks for several tools in
one turn (the briefing asks for four), the SDK runs them in parallel, so their start times in the trace can come
in another order. The dependency that matters, the price history before `calc_return`, is a separate turn and keeps its order.

In [21]:
import time
from phoenix.client import Client

time.sleep(3)   # give Phoenix a moment to ingest the last spans
spans = Client().spans.get_spans_dataframe(project_identifier="trajectory-evals")

roots = spans[spans["parent_id"].isna()].set_index("context.trace_id")      # one root span per run, named after the question
analyst_traces = spans[(spans["span_kind"] == "AGENT") & (spans["name"] == "Market Analyst")]["context.trace_id"]
tool_spans = spans[spans["span_kind"] == "TOOL"].sort_values("start_time")

print(len(analyst_traces), "analyst runs recorded in Phoenix (the judge and the reflector have their own traces)")
for trace_id in roots.loc[analyst_traces].sort_values("start_time").index:
    path = tool_spans[tool_spans["context.trace_id"] == trace_id]["name"].tolist()
    print(f"{roots.loc[trace_id, 'name'][:48]:48} {path}")

7 analyst runs recorded in Phoenix (the judge and the reflector have their own traces)
What is NVCH trading at right now?               ['get_quote']
What is NVCH trading at right now?               ['get_quote']
In one line: what is the news mood around HLXE t ['get_news']
Exactly how much has NVCH moved over the last 30 ['get_price_history', 'calc_return']
Pre-market briefing on ATLB: current price, valu ['get_quote', 'get_fundamentals', 'get_price_history', 'get_news']
Should the desk worry about HLXE's debt?         ['get_fundamentals']
Pre-market briefing on ATLB: current price, valu ['get_fundamentals', 'get_price_history', 'get_news', 'calc_return']


To reopen the traces later, without this notebook (PowerShell, from this folder):

```
$env:PHOENIX_WORKING_DIR = "$PWD\phoenix_data"
phoenix serve
```
Then http://localhost:6006, project **trajectory-evals**.

In [22]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(6006, height=1000)

<IPython.core.display.Javascript object>

## 11. Takeaways

1. **A right answer can hide a broken process.** Checking only the final answer would have passed the mental math.
2. **Grade the path with two checks.** A simple Python check (required tools, order) plus an LLM judge that knows the desk rules.
3. **Expect only what you can defend.** A list of required tools and one real dependency, not a full script of calls that fails good agents.
4. **A failure can become a lesson.** One general sentence in the prompt changed the behavior, without touching the code.
5. **Watch "cost-saving" lines in prompts.** "Use the fewest tool calls" quietly broke a desk rule nobody had written down for the agent.

*Educational content, not financial advice. All market data is fictional.*